# Original 69-Feature LightGBM Baseline

This diagnostic experiment compares LightGBM on the normalized original features against the 16-dimensional Autoencoder representation for S1 and S2.

In [ ]:
# Cell 1 - Locate the project root and import notebook dependencies.
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

project_root = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / 'configs' / 'config.yaml').exists()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f'Project root: {project_root}')

In [ ]:
# Cell 2 - Load configuration and verify original-feature plus target arrays.
from src.data_loading import load_config, resolve_project_path

config_path = project_root / 'configs' / 'config.yaml'
config = load_config(config_path)
processed_dir = resolve_project_path(config['paths']['processed_dir'], project_root)
metrics_dir = resolve_project_path(config['paths']['metrics_dir'], project_root)
figures_dir = resolve_project_path(config['paths']['figures_dir'], project_root)
required_paths = [
    processed_dir / 'X_train.npy',
    processed_dir / 'X_test.npy',
    processed_dir / 'y_train.npy',
    processed_dir / 'y_test.npy',
]
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f'Missing baseline input artifacts: {missing_paths}')

print(f"Original feature count: {config['data']['expected_feature_count']}")
print(f"Baseline scenarios: {config['baseline']['scenarios']}")

In [ ]:
# Cell 3 - Train missing original-feature models or reuse matching artifacts.
from src.baseline import run_original_feature_baseline

FORCE_RETRAIN_BASELINE = False
baseline_result = run_original_feature_baseline(
    config_path,
    force=FORCE_RETRAIN_BASELINE,
)
display(baseline_result['aggregate_comparison'])

In [ ]:
# Cell 4 - Verify that the controlled baseline changes only feature representation.
expected_iterations = int(config['lightgbm']['n_estimators'])
expected_features = int(config['data']['expected_feature_count'])
for scenario, report in baseline_result['training_reports'].items():
    assert report['representation'] == 'original_69', scenario
    assert report['feature_count'] == expected_features, scenario
    assert report['model_iterations'] == expected_iterations, scenario
    assert report['internal_validation'] is False, scenario
    assert report['all_configured_classes_predicted'], scenario
    if scenario == 's1_none':
        assert report['sample_weight_path'] is None
    if scenario == 's2_class_weight':
        assert report['sample_weight_path'] is not None

with open(metrics_dir / 'baseline_test_integrity.json', encoding='utf-8') as file:
    integrity = json.load(file)
assert integrity['test_artifacts_unchanged'] is True
assert integrity['test_hashes_before'] == integrity['test_hashes_after']
print('Controlled-baseline and test-integrity checks passed.')

In [ ]:
# Cell 5 - Compare DoS, DDoS, and XSS metrics across representations.
focus = baseline_result['focus_comparison']
display(focus)

focus_pivot = focus.pivot(
    index=['scenario', 'class_name'],
    columns='representation',
    values=['recall', 'f1_score', 'paired_confusion_rate'],
)
display(focus_pivot)

In [ ]:
# Cell 6 - Review the data-driven diagnosis for Autoencoder information loss.
diagnostic = baseline_result['diagnostic']
print(diagnostic['diagnosis'])
print()
print(diagnostic['causal_limit'])

best_f1 = pd.DataFrame(diagnostic['dos_ddos_best_f1_by_representation'])
display(best_f1)
display(pd.DataFrame([diagnostic['aggregate_s2_effect']]))
display(pd.DataFrame([diagnostic['xss_s2_effect']]))

In [ ]:
# Cell 7 - Display aggregate and confusion-matrix baseline figures.
display(Image(filename=baseline_result['figure_path']))
for scenario in config['baseline']['scenarios']:
    figure_path = figures_dir / f'confusion_matrix_original_69_{scenario}.png'
    if not figure_path.exists():
        raise FileNotFoundError(figure_path)
    display(Markdown(f'## Original 69 / {scenario}'))
    display(Image(filename=str(figure_path)))

In [ ]:
# Cell 8 - List baseline artifacts prepared for the thesis discussion.
artifact_paths = [
    Path(baseline_result['aggregate_path']),
    Path(baseline_result['focus_path']),
    Path(baseline_result['diagnostic_path']),
    Path(baseline_result['figure_path']),
    Path(baseline_result['markdown_path']),
]
artifact_paths.extend(
    Path(report['model_path'])
    for report in baseline_result['training_reports'].values()
)
artifact_summary = pd.DataFrame({
    'artifact': [str(path) for path in artifact_paths],
    'exists': [path.exists() for path in artifact_paths],
})
assert artifact_summary['exists'].all()
display(artifact_summary)
print('Original-feature baseline artifacts are complete.')